# معمل التجربة: استخدام مكتبة FAISS المحلية من ميتا

مكتبة FAISS (Facebook AI Similarity Search) هي الحل المفضل لما تكون عايز تشغل RAG بالكامل على جهازك أو السيرفر الخاص بيك بدون تكلفة وبسرعة خيالية.

في المعمل ده هنجرب:
1. بناء الفهرس في الذاكرة.
2. حفظ الفهرس كملف على القرص (`save_local`).
3. إعادة تحميل الفهرس واستخدامه في البحث لاحقاً (`load_local`).

### 1. بناء الفهرس وحفظه على القرص
يلا نجهز الفهرس ونسيفه محلياً:

In [1]:
import tempfile
import shutil
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
temp_dir = tempfile.mkdtemp()

docs = [
    Document(page_content="مكتبة FAISS مصممة للبحث السريع في المتجهات الضخمة بكفاءة عالية على المعالجات المركزية وكروت الشاشة."),
    Document(page_content="فهرس IVF يقسم فضاء المتجهات لمجموعات لتسريع الاسترجاع."),
    Document(page_content="فهرس HNSW يوفر دقة وسرعة استثنائية في البحث التقريبي لأقرب الجيران."),
]

# بناء الفهرس وحفظه
vector_store = FAISS.from_documents(docs, embeddings)
vector_store.save_local(temp_dir)
print(f"تم حفظ فهرس FAISS بنجاح في: {temp_dir}")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_11124\4084973138.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

تم حفظ فهرس FAISS بنجاح في: C:\Users\ALIASH~1\AppData\Local\Temp\tmp80juvyxo


### 2. إعادة تحميل الفهرس المحفوظ والبحث فيه
تعال نقرأ الفهرس ونبحث بدرجات المسافة:

In [2]:
# إعادة التحميل
loaded_store = FAISS.load_local(temp_dir, embeddings, allow_dangerous_deserialization=True)

query = "ما هو الفهرس الذي يوفر دقة عالية باستخدام الرسوم البيانية HNSW؟"
results_with_scores = loaded_store.similarity_search_with_score(query, k=1)

for doc, score in results_with_scores:
    print(f"المسافة (Distance): {score:.4f}")
    print(f"المحتوى المسترجع: {doc.page_content}")

shutil.rmtree(temp_dir, ignore_errors=True)

المسافة (Distance): 0.2309
المحتوى المسترجع: فهرس HNSW يوفر دقة وسرعة استثنائية في البحث التقريبي لأقرب الجيران.


### 3. جرب بنفسك!
جرب تقارن بين مسافات الجمل المتشابهة والمختلفة وشوف الرقم بيتغير إزاي.